# CS781 Phase 1 — Load the 1,000-step model and generate predictions

This is a trimmed inference-only notebook extracted from the provided notebook. It does **not** retrain the model.

## Before running
1. Attach the Phase-1 artifacts dataset containing `tokenizer/legal_bpe_32k.json` and the test data.
2. Make the trained checkpoint `checkpoint_latest.pt` available in `/kaggle/input/...` (recommended) or `/kaggle/working/phase1_decoder_transformer/checkpoints/`.
3. Run the cells from top to bottom.

The checkpoint is not embedded in this notebook. If the original Kaggle session has ended, `/kaggle/working` may have been reset, so upload/attach the saved checkpoint first.


In [ ]:
# 1. Imports and runtime
import os, json, zipfile, math
from pathlib import Path

import torch
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer, decoders

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
CONTEXT_LENGTH = 4096


In [ ]:
# 2. Locate Phase-1 tokenizer/artifacts and trained checkpoint
ARTIFACT_CANDIDATES = [
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"),
]
ARTIFACT_ROOT = next(
    (p for p in ARTIFACT_CANDIDATES if (p / "tokenizer/legal_bpe_32k.json").exists()),
    None
)

if ARTIFACT_ROOT is None:
    matches = list(Path("/kaggle/input").rglob("legal_bpe_32k.json"))
    assert matches, "Could not find legal_bpe_32k.json. Attach the Phase-1 artifacts dataset."
    TOKENIZER_PATH = matches[0]
else:
    TOKENIZER_PATH = ARTIFACT_ROOT / "tokenizer/legal_bpe_32k.json"

checkpoint_candidates = [
    Path("/kaggle/working/phase1_decoder_transformer/checkpoints/checkpoint_latest.pt"),
]
checkpoint_candidates += list(Path("/kaggle/input").rglob("checkpoint_latest.pt"))
checkpoint_candidates += list(Path("/kaggle/input").rglob("checkpoint_step_001000.pt"))

FINAL_CKPT = next((p for p in checkpoint_candidates if p.exists()), None)
assert FINAL_CKPT is not None, (
    "Trained checkpoint not found. Upload/attach checkpoint_latest.pt or "
    "checkpoint_step_001000.pt to Kaggle input, or place it at "
    "/kaggle/working/phase1_decoder_transformer/checkpoints/checkpoint_latest.pt"
)

print("Tokenizer:", TOKENIZER_PATH)
print("Checkpoint:", FINAL_CKPT)


In [ ]:
# 3. Load tokenizer
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel()

VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Vocabulary:", VOCAB_SIZE)
print("PAD/BOS/EOS:", PAD_ID, BOS_ID, EOS_ID)
assert VOCAB_SIZE == 32000
assert PAD_ID is not None and BOS_ID is not None and EOS_ID is not None


In [ ]:
# 4. Rebuild the same GPT-2-style architecture and load trained weights
# weights_only=False is needed for checkpoints containing optimizer/scaler metadata
try:
    checkpoint = torch.load(FINAL_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(FINAL_CKPT, map_location="cpu")

cfg = checkpoint.get("model_config", {})
hf_config = GPT2Config(
    vocab_size=int(cfg.get("vocab_size", VOCAB_SIZE)),
    n_positions=int(cfg.get("context_length", CONTEXT_LENGTH)),
    n_ctx=int(cfg.get("context_length", CONTEXT_LENGTH)),
    n_embd=int(cfg.get("hidden_size", 768)),
    n_layer=int(cfg.get("num_layers", 12)),
    n_head=int(cfg.get("num_heads", 12)),
    n_inner=int(cfg.get("hidden_size", 768)) * 4,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    use_cache=False,
)

model = GPT2LMHeadModel(hf_config)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded checkpoint step:", checkpoint.get("step", "unknown"))
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Device:", DEVICE)


In [ ]:
# 5. Locate and load Phase-1 test.json
test_candidates = []
for p in Path("/kaggle/input").rglob("test.json"):
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if (
            isinstance(obj, dict) and len(obj) > 0
            and all(isinstance(v, dict) and isinstance(v.get("text"), str)
                    for v in list(obj.values())[:min(10, len(obj))])
        ):
            test_candidates.append((p, obj))
    except Exception:
        pass

assert test_candidates, (
    "Could not find Phase-1 test.json in /kaggle/input. "
    "Attach the dataset containing the official test.json."
)

TEST_JSON, test_data = test_candidates[0]
print("Using:", TEST_JSON)
print("Test examples:", len(test_data))
print("Example ID:", next(iter(test_data)))


In [ ]:
# 6. Generation function used by the original notebook
def generate_128_words(prompt, temperature=0.8, top_k=50, max_generation_tokens=300):
    encoded = tokenizer.encode(prompt)
    prompt_ids = encoded.ids

    if not prompt_ids:
        prompt_ids = [BOS_ID]
    if len(prompt_ids) >= CONTEXT_LENGTH:
        prompt_ids = prompt_ids[-CONTEXT_LENGTH + 1:]

    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_generation_tokens):
            context_ids = generated[:, -CONTEXT_LENGTH:]
            attention_mask = torch.ones_like(context_ids)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                outputs = model(
                    input_ids=context_ids,
                    attention_mask=attention_mask,
                    use_cache=False
                )
                logits = outputs.logits[:, -1, :]

            logits = logits / temperature
            if top_k is not None:
                k = min(top_k, logits.shape[-1])
                values, indices = torch.topk(logits, k=k, dim=-1)
                filtered = torch.full_like(logits, float("-inf"))
                filtered.scatter_(1, indices, values)
                logits = filtered

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim=1)

            new_ids = generated[0, input_ids.shape[1]:].tolist()
            text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            if len(text.split()) >= 128:
                break

    new_ids = generated[0, input_ids.shape[1]:].tolist()
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    text = " ".join(text.split())
    return " ".join(text.split()[:128])


In [ ]:
# 7. Quick smoke test on one example
first_id = next(iter(test_data))
sample_output = generate_128_words(test_data[first_id]["text"])

print("ID:", first_id)
print("Generated words:", len(sample_output.split()))
print(sample_output[:1000])
assert len(sample_output.split()) == 128
assert "Ġ" not in sample_output and "Ċ" not in sample_output


In [ ]:
# 8. Generate predictions for all test examples
predictions = {}
total = len(test_data)

for i, (test_id, item) in enumerate(test_data.items(), start=1):
    continuation = generate_128_words(item["text"])
    predictions[test_id] = {"text": continuation}

    if i % 10 == 0 or i == total:
        print(f"[{i}/{total}] ID={test_id} | words={len(continuation.split())}")

assert set(predictions) == set(test_data)
for test_id, item in predictions.items():
    assert set(item) == {"text"}
    assert isinstance(item["text"], str)
    assert len(item["text"].split()) == 128
    assert "Ġ" not in item["text"] and "Ċ" not in item["text"]

print("Validated prediction count:", len(predictions))


In [ ]:
# 9. Save prediction.json and ZIP submission
SUBMISSION_DIR = Path("/kaggle/working/phase1_decoder_transformer_submission")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = SUBMISSION_DIR / "prediction.json"
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

ZIP_PATH = Path("/kaggle/working/phase1_decoder_transformer_submission.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_PATH, arcname="prediction.json")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    assert z.namelist() == ["prediction.json"]

print("Prediction file:", PREDICTION_PATH)
print("Submission ZIP:", ZIP_PATH)
